# PROJECT SETUP AND CONFIGURATION

**1. IMPORTS**

In [1]:
import random
import json
import time
import os
import uuid

from datetime import datetime, timezone
from google import genai
from getpass import getpass

**2. GEMINI CLIENT CONFIGURATION**

In [2]:
# Load the API key from environment variables
# If it isn't available, ask the user to enter it securely
if not os.environ.get('GEMINI_API_KEY'):
    os.environ['GEMINI_API_KEY']= getpass.getpass('Enter Gemini API key: ')

# Initialize the Gemini API client
client = genai.Client()
print('Gemini client initialized successfully!')

Enter Gemini API key:  ········


Gemini client initialized successfully!


**3. MODEL CONFIGURATION**

In [3]:
# Model used for puzzle generation and AI reasoning
MODEL_NAME = "gemini-3.5-flash-lite"

print('Selected model: ', MODEL_NAME)

Selected model:  gemini-3.5-flash-lite


**4. GAME CONFIGURATIONS**

In [4]:
# Puzzle categories supported in the current MVP
PUZZLE_CATEGORIES = ['number_pattern','riddle']

# Available difficulty levels, ordered from easiest to hardest
DIFFICULTY_LEVELS = ['easy','medium','hard']

# Every new game session starts at Easy
STARTING_DIFFICULTY = 'easy'

# Maximum number of hints available for each puzzle
MAX_HINTS = 3

# Number of recent puzzles to consider when avoiding repetition
RECENT_HISTORY_LIMIT = 10

# Maximum attempts to generate a puzzle that passes validation
MAX_GENERATION_ATTEMPTS = 3

# PUZZLE DATA STRUCTURE

**5. COMMON PUZZLE DATA STRUCTURE**

In [5]:
def create_puzzle(
    category,
    difficulty,
    question,
    content,
    answer_type,
    correct_answer,
    hints,
    explanation,
    options=None,
    accepted_answers=None,
    source="ai"
):
    """
    Creates a standardized puzzle dictionary.

    All puzzle categories share the same outer structure.
    Category-specific information is stored inside 'content'.

    Parameters:
        category (str): Puzzle category.
        difficulty (str): Difficulty level.
        question (str): Question displayed to the player.
        content (dict): Category-specific puzzle information.
        answer_type (str): Expected answer type.
        correct_answer: The correct answer.
        hints (list): Three progressive hints.
        explanation (str): Explanation of the solution.
        options (list, optional): Multiple-choice options.
        accepted_answers (list, optional): Alternative valid answers.
        source (str): Where the puzzle originated.

    Returns:
        dict: A standardized puzzle object.
    """

    # Generate a unique identifier for each puzzle.
    puzzle_id = str(uuid.uuid4())

    # Create the standardized puzzle structure.
    puzzle = {
        # Identification and classification
        "puzzle_id": puzzle_id,
        "category": category,
        "difficulty": difficulty,

        # Puzzle content
        "question": question,
        "content": content,

        # Answer information
        "answer_type": answer_type,
        "correct_answer": correct_answer,
        "options": options if options is not None else [],
        "accepted_answers": (
            accepted_answers if accepted_answers is not None else []
        ),

        # Player assistance
        "hints": hints,
        "explanation": explanation,

        # Metadata
        "source": source,
        "validation_status": "pending"
    }

    return puzzle

# PUZZLE GENERATION

**6. DIFFICULTY GUIDELINES**

In [6]:
# Guidelines used by Gemini to control puzzle complexity.

PUZZLE_DIFFICULTY_GUIDELINES = {
    "number_pattern": {
        "easy": (
            "Use straightforward patterns such as constant addition, "
            "subtraction, multiplication, or division."
        ),
        "medium": (
            "Use moderately challenging patterns such as alternating "
            "operations, increasing differences, or combinations "
            "of simple operations."
        ),
        "hard": (
            "Use challenging patterns such as multiple interacting "
            "sequences, complex difference patterns, or less obvious "
            "mathematical relationships."
        )
    },

    "riddle": {
        "easy": (
            "Use familiar objects, everyday situations, and "
            "straightforward clues. The answer should be relatively "
            "easy to infer."
        ),
        "medium": (
            "Use more indirect clues, wordplay, and multiple "
            "reasoning steps. The answer should require some thought."
        ),
        "hard": (
            "Use challenging wordplay, layered clues, and less obvious "
            "connections. Avoid obscure trivia or knowledge players "
            "would need to look up."
        )
    }
}

**7. GEMINI RESPONSE HANDLING**

In [7]:
def generate_json_response(prompt):
    """
    Sends a prompt to Gemini and returns the response as a Python dictionary.

    Parameters:
        prompt (str): Instructions sent to the Gemini model.

    Returns:
        dict: Parsed JSON response from Gemini.

    Raises:
        ValueError: If Gemini returns an empty or invalid JSON response.
    """

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config={
            "response_mime_type": "application/json"
        }
    )

    # Extract the text response.
    response_text = response.text

    # Ensure Gemini returned a response.
    if not response_text:
        raise ValueError("Gemini returned an empty response.")

    # Convert JSON text into a Python dictionary.
    try:
        return json.loads(response_text)

    except json.JSONDecodeError as error:
        raise ValueError(
            f"Gemini returned invalid JSON: {response_text}"
        ) from error

**8. NUMBER PATTERN PUZZLE**

In [8]:
# ==========================================
# NUMBER PATTERN PROMPT
# ==========================================

def get_number_pattern_prompt(difficulty, recent_puzzles=None, feedback=None):

    """
    Builds the prompt used to generate a number pattern puzzle.

    Parameters:
        difficulty (str): Requested difficulty.
        recent_puzzles (list): Recent puzzles to avoid repeating.
        feedback (str, optional): Validation errors from a previous attempt.

    Returns:
        str: Prompt for Gemini.
    """

    # Use an empty list if no puzzle history is provided
    recent_puzzles = recent_puzzles or []

    # Retrieve the guidelines for the requested difficulty
    difficulty_guidelines = PUZZLE_DIFFICULTY_GUIDELINES["number_pattern"][difficulty]

    prompt = f"""
    You are an expert puzzle creator for Puzzle Pirates, an adaptive brain puzzle game.
    
    Generate ONE original number pattern puzzle.
    
    DIFFICULTY: {difficulty}
    difficulty_guidelines = {difficulty_guidelines}
    
    PUZZLE REQUIREMENTS:
    
    1. The puzzle must have exactly one missing number.
    2. The sequence must contain at least 4 numbers.
    3. The missing number must be represented as null.
    4. Use a clear and mathematically consistent pattern.
    5. The correct answer must be an integer.
    6. Provide exactly 4 multiple-choice options.
    7. Only one option can be correct.
    8. Provide exactly 3 progressive hints.
    9. Hints must gradually reveal the solving approach without immediately giving away the answer.
    10. Provide a clear explanation of the solution.
    11. Avoid ambiguous sequences with multiple valid answers.
    12. Match the complexity of the requested difficulty.
    
    VARIETY REQUIREMENTS:
    
    - Use different mathematical patterns.
    - Vary the starting numbers and sequence length.
    - Explore addition, subtraction, multiplication, division, alternating patterns, and other valid patterns.
    - Avoid repeatedly generating simple arithmetic sequences.
    - Do not copy the examples or previously generated puzzles.
    
    RECENTLY GENERATED PUZZLES TO AVOID: 
    {json.dumps(recent_puzzles, indent=2)}
    
    Generate something substantially different from these puzzles.
    
    Return ONLY valid JSON using this structure:
    {{
        "category": "number_pattern",
        "difficulty": "{difficulty}",
        "question": "Find the missing number.",
        "content": {{
            "sequence": [3, 6, 9, null, 15],
            "missing_position": 3,
            "pattern_type": "addition"
        }},
        "answer_type": "integer",
        "correct_answer": 12,
        "options": [10, 11, 12, 14],
        "accepted_answers": [],
        "hints": [
            "Hint 1",
            "Hint 2",
            "Hint 3"
        ],
        "explanation": "Explain the pattern and answer."
    }}

    The example above is only a format reference. Generate a new puzzle rather than copying it.
    """

    # Add validation feedback if the previous attempt failed
    if feedback:
        prompt += f"""
    
    PREVIOUS VALIDATION ERRORS:
    {feedback}
    
    Generate a new puzzle and correct all the issues above
    """

    return prompt

In [9]:
# ==========================================
# NUMBER PATTERN GENERATOR
# ==========================================

def generate_number_pattern(difficulty, recent_puzzles=None, feedback=None):

    """
    Generates a number pattern puzzle using Gemini.

    Parameters:
        difficulty (str): Requested difficulty.
        recent_puzzles (list, optional): Puzzles to avoid repeating.
        feedback (str, optional): Validation feedback.

    Returns:
        dict: Raw puzzle data returned by Gemini.
    """

    # Build the generation prompt 
    prompt = get_number_pattern_prompt(
        difficulty,
        recent_puzzles = recent_puzzles,
        feedback=feedback
    )

    # Generate and parse the JSON response
    generated_data = generate_json_response(prompt)

    return generated_data


**9. RIDDLE PUZZLE**

In [10]:
# ==========================================
# RIDDLE PROMPT
# ==========================================

def get_riddle_prompt(difficulty, recent_puzzles=None, feedback=None):

    """
    Builds the prompt used to generate a riddle.

    Parameters:
        difficulty (str): Requested difficulty.
        recent_puzzles (list): Recent riddles to avoid repeating.
        feedback (str, optional): Validation errors from a previous attempt.

    Returns:
        str: Prompt for Gemini.
    """

    # Use an empty list if no puzzle history is provided
    recent_puzzles = recent_puzzles or []

    # Retrieve difficulty-specific guidelines
    difficulty_guidelines = PUZZLE_DIFFICULTY_GUIDELINES['riddle'][difficulty]

    prompt = f"""
    You are an expert riddle creator for Puzzle Pirates, an adaptive brain puzzle game.

    Generate ONE original riddle.

    DIFFICULTY: {difficulty}
    difficulty_guidelines = {difficulty_guidelines}

    PUZZLE REQUIREMENTS:

    1. Generate one clear and engaging riddle.
    2. The riddle must have one intended answer.
    3. Avoid ambiguous clues that support multiple unrelated answers.
    4. Provide the correct answer as a short text string.
    5. Provide alternative accepted answers where appropriate.
    6. Do not include the correct answer in the question.
    7. Generate exactly 3 progressive hints.
    8. Each hint should reveal more information than the previous one.
    9. Hints must not immediately reveal the answer.
    10. Provide a clear explanation of the answer.
    11. Avoid obscure trivia and culturally specific references.
    12. Generate an original riddle rather than copying common examples.

    VARIETY REQUIREMENTS:

    - Explore different themes, objects, situations, and types of wordplay.
    - Avoid repeatedly using familiar riddles.
    - Avoid generating riddles similar to the recent examples below.

    RECENTLY GENERATED RIDDLES TO AVOID:
    {json.dumps(recent_puzzles, indent=2)}

    Return ONLY valid JSON with these fields:
    {{
        "category": "riddle",
        "difficulty": "{difficulty}",
        "question": "The complete riddle text",
        "content": {{
            "riddle_text": "The complete riddle text"
        }},
        "answer_type": "string",
        "correct_answer": "intended answer",
        "options": [],
        "accepted_answers": ["alternative answer"],
        "hints": [
            "First hint",
            "Second hint",
            "Third hint"
        ],
        "explanation": "Explain why this is the answer."
    }}

    The example is only a format reference. Do not copy it. 
    Do not include Markdown or additional text.
    
    """

    # Include validation feedback when correcting a failed generation
    if feedback:
        prompt += f"""
            PREVIOUS VALIDATION ERRORS: {feedback}

            Generate a new riddle that corrects these issues. 
        """

    return prompt

In [11]:
# ==========================================
# RIDDLE GENERATOR
# ==========================================

def generate_riddle(difficulty, recent_puzzles=None, feedback=None):

    """
    Generates a riddle using Gemini.

    Parameters:
        difficulty (str): Requested difficulty.
        recent_puzzles (list, optional): Riddles to avoid repeating.
        feedback (str, optional): Validation feedback.

    Returns:
        dict: Raw riddle data returned by Gemini.
    """

    # Build the riddle generation prompt
    prompt = get_riddle_prompt(
        difficulty = difficulty,
        recent_puzzles = recent_puzzles,
        feedback = feedback
    )

    # Generate and parse the JSON response
    generated_data = generate_json_response(prompt)

    return generated_data

# PUZZLE VALIDATION

**10. COMMON VALIDATION**

In [12]:
def validate_common_fields(puzzle):

    """
    Validates fields shared by all puzzle categories.

    Parameters:
        puzzle (dict): Puzzle object to validate.

    Returns:
        list: Validation error messages.
               Empty list means no common validation errors.
    """
    
    errors = []

    # All puzzles must contain these fields
    required_fields = [
        "puzzle_id",
        "category",
        "difficulty",
        "question",
        "content",
        "answer_type",
        "correct_answer",
        "options",
        "accepted_answers",
        "hints",
        "explanation",
        "source",
        "validation_status"
    ]

    # Check for missing fields before accessing them
    missing_fields = [
        field for field in required_fields
        if field not in puzzle
    ]

    if missing_fields:
        errors.extend([f"Missing required field: {field}" for field in missing_fields])

        # Stop here because further checks may require missing fields
        return errors

    # Category must be one of the supported categories
    if puzzle['category'] not in PUZZLE_CATEGORIES:
        errors.append('Invalid puzzle category')

    # Difficulty must be a supported level
    if puzzle['difficulty'] not in DIFFICULTY_LEVELS:
        errors.append('Invalid difficulty level.')

    # Question must be a supported level
    if not isinstance(puzzle['question'], str) or not puzzle['question'].strip():
        errors.append('Question must be a non-empty string')

    # Question must contain readable text
    if not isinstance(puzzle['question'], str) or not puzzle['question'].strip():
        errors.append('Question must be a non-empty string')

    # Content must be a dictionary
    if not isinstance(puzzle['content'], dict):
        errors.append('Content must be a dictionary')

    # Exactly 3 progressive hints are required
    hints = puzzle['hints']

    if not isinstance(hints, list) or len(hints) != MAX_HINTS:
        errors.append(f"Each puzzle must contain exactly {MAX_HINTS} hints")

    elif not all (isinstance(hint, str) and hint.strip() for hint in hints):
        errors.append('Every hint must be a non-empty string')

    # Options must be stored as a list
    if not isinstance(puzzle['options'], list):
        errors.append('Options must be a list')

    # Accepted answers must be stored as a list
    if not isinstance(puzzle['accepted_answers'], list):
        errors.append('Accepted answers must be a list')

    # Explanation must be readable text
    if not isinstance(puzzle['explanation'], str) or not puzzle['explanation'].strip():
        errors.append('Explanation must be a non-empty string')
    return errors
    

**11. NUMBER PATTERN VALIDATION**

In [13]:
def validate_number_pattern(puzzle):
    """
    Validates the structure and answer of a number pattern puzzle.

    Checks:
        - Required category-specific content.
        - Sequence structure and values.
        - Exactly one missing element.
        - Correct missing position.
        - Answer type and value.
        - Multiple-choice options.
        - Accepted answers.

    Parameters:
        puzzle (dict): Number pattern puzzle.

    Returns:
        list: Validation errors. An empty list means valid.
    """
    
    errors = []

    # Retrieve category-specific content safely
    content = puzzle.get("content")

    if not isinstance(content, dict):
        return ['Number pattern content must be a dictionary']

    # Required fields inside number pattern content
    required_content_fields = [
        "sequence",
        "missing_position",
        "pattern_type"
    ]

    # Check that all required content fields exist
    missing_fields = [
        field for field in required_content_fields
        if field not in content
    ]

    if missing_fields:
        errros.extend([f"Number pattern is missing: {field}" for field in missing_fields])
        return errors

    sequence = content['sequence']
    missing_position = content['missing_position']
    pattern_type = content['pattern_type']

    # ------------------------------------------
    # Sequence validation
    # ------------------------------------------

    if not isinstance(sequence, list):
        errors.append('Sequence must be a list.')

    else:
        # A sequence must have enough positions to form a pattern.
        if len(sequence) < 4:
            errors.append('Sequence must contain at least 4 positions')

        #Sequence values must be integers or None
        if not all(
            value is None or (isinstance(value, int) and not isinstance(value, bool))
            for value in sequence
        ):
            errors.append('Sequence values must be integers or None')

        # Exactly one position must be missing
        if sequence.count(None) != 1:
            errors.append('Sequence must contain exactly one missing value')

    # ------------------------------------------
    # Missing position validation
    # ------------------------------------------

    if not isinstance(missing_position, int) or isinstance(missing_position, bool):
        errors.append('Missing position must be an integer')

    elif isinstance(sequence, list):
        # The index must be within the sequence bounds
        if not (0 <= missing_position < len(sequence)):
            errors.append('Missing position is out of range')

        # The indicated position must contain None
        elif sequence[missing_position] is not None:
            errors.append('Specified missing position must contain None.')

    # ------------------------------------------
    # Pattern type validation
    # ------------------------------------------

    if not isinstance(pattern_type, str) or not pattern_type.strip():
        errors.append('Pattern type must be non-empty text')

    # ------------------------------------------
    # Answer validation
    # ------------------------------------------

    if puzzle.get('answer_type') != 'integer':
        errors.append('Number pattern answer type must be integer')

    answer = puzzle.get('correct_answer')

    # Booleans are technically subclasses of integers in Python, so we explicitly exclude them
    if not isinstance(answer, int) or isinstance(answer, bool):
        errors.append('Correct answer must be an integer')

    # ------------------------------------------
    # Options validation
    # ------------------------------------------

    options = puzzle.get('options')

    if not isinstance(options, list):
        errors.append('Options must be a list')

    else:
        if len(options) != 4:
            errors.append('Number pattern must have exactly 4 options')

        # Ensure the options are distinct
        if len(set(options)) != len(options):
            errors.append('Options must be unique')

        # The correct answer must be one of the choices
        if answer not in options:
            errors.append('Correcr answer must be included in the options')

        # Every option must be an integer
        if not all (isinstance(option, int) and not isinstance(option, bool)
                   for option in options):
            errors.append('All options must be integers')

    # ------------------------------------------
    # Accepted answers validation
    # ------------------------------------------

    accepted_answers = puzzle.get('accepted_answers')

    if accepted_answers != []:
        errors.append('Number pattern accepted_answers must be empty')

    return errors


**12. RIDDLE VALIDATION**

In [14]:
def validate_riddle(puzzle):

    """
    Validates the structure and answer of a riddle.

    Checks:
        - Required category-specific content.
        - Riddle text exists.
        - Answer is a non-empty string.
        - Accepted answers are valid strings.
        - No duplicate accepted answers.
        - Options are empty.
        - Common puzzle fields are valid.

    Parameters:
        puzzle (dict): Riddle puzzle.

    Returns:
        list: Validation errors. An empty list means valid.
    """

    errors = []

    # Retrieve category-specific content safely
    content = puzzle.get('content')

    if not isinstance(content, dict):
        return ['Riddle content must be dictionary']

    # ------------------------------------------
    # Riddle text validation
    # ------------------------------------------

    riddle_text = content.get('riddle_text')

    if not isinstance(riddle_text, str) or not riddle_text.strip():
        errors.append('Riddle text must be a non-empty string')

    # ------------------------------------------
    # Answer validation
    # ------------------------------------------  

    if puzzle.get("answer_type") != "string":
        errors.append('Riddle answer type must be string')

    answer = puzzle.get('correct_answer')

    if not isinstance(answer, str) or not answer.strip():
        errors.append('Correct answer must be a non-empty string')

    # ------------------------------------------
    # Accepted answers validation
    # ------------------------------------------

    accepted_answers = puzzle.get('accepted_answers')

    if not isinstance(accepted_answers, list):
        errors.append('Accepted answers must be a list')

    else:
        # Every alternative answer must be non-empty text
        if not all(isinstance(value, str) and value.strip() 
                  for value in accepted_answers):
            errors.append('Accepted answers must contain only non-empty strings')

        # Detect duplicates, ignoring case and whitespace
        normalized_answers = [
            value.strip().lower()
            for value in accepted_answers
            if isinstance(value, str)
        ]

        if len(normalized_answers) != len(set(normalized_answers)):
            errors.append('Accepted answers must not contain duplicates')

        # The correct answer should not be repeated among accepted answers
        if isinstance(answer, str):
            normalized_correct_answer = answer.strip().lower()

            if normalized_correct_answer in normalized_answers:
                errors.append('Accepted answers must not duplicate the correct answer')
        
    # ------------------------------------------
    # Options validation
    # ------------------------------------------

    options = puzzle.get('options')

    if options != []:
        errors.append('Riddle options must be empty')

    return errors


**13. UNIFIED PUZZLE VALIDATION**

In [15]:
def validate_puzzle(puzzle):
    """
    Validates a puzzle using common and category-specific rules.

    Parameters:
        puzzle (dict): Puzzle object to validate.

    Returns:
        list: All validation errors.
              An empty list means the puzzle is valid.
    """

    # Validate fields shared by all puzzle categories
    errors = validate_common_fields(puzzle)

    # If the category is missing or invalid, stop category validation
    category = puzzle.get('category')

    if category not in PUZZLE_CATEGORIES:
        return errors

    # Run category-specific validation
    if category == 'number_pattern':
        errors.extend(validate_number_pattern(puzzle))

    elif category == 'riddle':
        errors.extend(validate_riddle(puzzle))

    return errors

# PUZZLE MANAGEMENT

In [16]:
# Stored metadata about successfully generated puzzles
# Used to reduce repitition across future generation
generated_puzzle_history = []

**14. CATEGORY-BASED PUZZLE GENERATION**

In [17]:
def generate_puzzle_by_category(category, difficulty, feedback=None):

    """
    Routes puzzle generation to the appropriate generator.

    Parameters:
        category (str): Puzzle category.
        difficulty (str): Requested difficulty.
        feedback (str, optional): Validation feedback.

    Returns:
        dict: Raw generated puzzle data.
    """

    # Retrieve recent puzzle history to help avoid repitition
    recent_puzzles = [puzzle 
                     for puzzle in generated_puzzle_history[-RECENT_HISTORY_LIMIT:]
                     if puzzle['category'] == category]

    # Select the generator based on puzzle category
    if category == "number_pattern":
        return generate_number_pattern(
            difficulty = difficulty, 
            recent_puzzles = recent_puzzles,
            feedback = feedback
        )

    elif category == "riddle":
        return generate_riddle(
            difficulty = difficulty,
            recent_puzzles = recent_puzzles,
            feedback = feedback
        )

    else:
        raise ValueError(f"Unsupported puzzle category: {category}")

**15. NORMALIZE GEMINI OUTPUT**

In [18]:
def convert_to_puzzle(generated_data):

    """
    Converts raw Gemini output into the common puzzle structure.

    Missing fields receive safe defaults so the validator can
    report them as validation errors instead of causing KeyErrors.

    Parameters:
        generated_data (dict): Raw Gemini response.

    Returns:
        dict: Standardized puzzle object.
    """

    return create_puzzle(
        category=generated_data.get("category", ""),
        difficulty=generated_data.get("difficulty", ""),
        question=generated_data.get("question", ""),
        content=generated_data.get("content", {}),
        answer_type=generated_data.get("answer_type", ""),
        correct_answer=generated_data.get("correct_answer"),
        hints=generated_data.get("hints", []),
        explanation=generated_data.get("explanation", ""),
        options=generated_data.get("options", []),
        accepted_answers=generated_data.get("accepted_answers", []),
        source="ai"
    )


**16. VALIDATED PUZZLE GENERATION PIPELINE**

In [19]:
def generate_validated_puzzle(category, difficulty):
    """
    Generates a puzzle and retries until it passes validation
    or the maximum number of attempts is reached.

    Parameters:
        category (str): Puzzle category.
        difficulty (str): Requested difficulty.

    Returns:
        dict: A validated puzzle.

    Raises:
        RuntimeError: If no valid puzzle is generated.
    """

    feedback = None

    # Retry generation when validation fails.
    for attempt in range(1, MAX_GENERATION_ATTEMPTS + 1):

        print(
            f"Generation attempt {attempt}/"
            f"{MAX_GENERATION_ATTEMPTS}"
        )

        try:
            # Generate raw puzzle data.
            generated_data = generate_puzzle_by_category(
                category=category,
                difficulty=difficulty,
                feedback=feedback
            )

            # Normalize the raw response into our common structure.
            puzzle = convert_to_puzzle(generated_data)

            # Validate the standardized puzzle.
            errors = validate_puzzle(puzzle)

        except Exception as error:
            # Treat generation or conversion errors as retry feedback.
            errors = [f"Generation error: {str(error)}"]
            puzzle = None

        # ------------------------------------------
        # Successful generation
        # ------------------------------------------

        if not errors:
            puzzle["validation_status"] = "valid"

            print("Puzzle generated and validated successfully!")

            # Store compact metadata to reduce repetition.
            history_entry = {
                "puzzle_id": puzzle["puzzle_id"],
                "category": puzzle["category"],
                "difficulty": puzzle["difficulty"],
                "question": puzzle["question"],
                "content": puzzle["content"],
                "correct_answer": puzzle["correct_answer"]
            }

            generated_puzzle_history.append(history_entry)

            return puzzle

        # ------------------------------------------
        # Failed validation
        # ------------------------------------------

        print("Validation failed:")

        for error in errors:
            print("-", error)

        # Send validation errors back to Gemini on the next attempt.
        feedback = "\n".join(errors)

    # ------------------------------------------
    # All attempts exhausted
    # ------------------------------------------

    raise RuntimeError(
        f"Failed to generate a valid {category} puzzle "
        f"at {difficulty} difficulty after "
        f"{MAX_GENERATION_ATTEMPTS} attempts."
    )

# GAME SESSION MANAGEMENT

**17. GAME SESSION**

Each time a player starts playing:
- A new session is created.
- Difficulty starts at Easy.
- The player has no previous performance history.
- A category is selected randomly.
- The session tracks progress.

In [20]:
#defining game session

def create_game_session():

    """
    Create and initialize a new game session.

    Every new session starts at easy difficulty with zero score
    and no puzzles played.
    """

    session = {
        'session_id': str(uuid.uuid4()),
        'started_at': datetime.now(timezone.utc).isoformat(),
        'current_difficulty': STARTING_DIFFICULTY,
        
        'total_attempts': 0,
        'correct_answers': 0,
        'incorrect_answers': 0,

        'total_time': 0,
        'total_hints_used':0,

        'attempt_history': [],
        'puzzle_history': [],
        'adaptation_history': []
    }

    return session


**18. CATEGORY SELECTION**

- Categories are selected randomly.
- The same category won't appear twice consecutively.

In [21]:
def select_random_category(session):
    """
    Randomly select a puzzle category, avoiding the category
    used in the previous puzzle.
    """

    previous_category = None

    # Get the category of the previous puzzle, if one exists.
    if session["puzzle_history"]:
        previous_category = session["puzzle_history"][-1]["category"]

    # Exclude the previous category from the available choices.
    available_categories = [
        category
        for category in PUZZLE_CATEGORIES
        if category != previous_category
    ]

    # Randomly select one of the remaining categories.
    selected_category = random.choice(available_categories)

    return selected_category

**19. START NEXT ROUND**

In [48]:
def start_next_round(session):

    """
    Select a category, generate a validated puzzle at the session's current difficulty,
    and record it in the session.
    """

    # Select a category while avoiding consecutive repetitions
    category = select_random_category(session)

    # Get the current difficulty from the session
    difficulty = session['current_difficulty']

    # Generate a puzzle and ensure it passes validation
    puzzle = generate_validated_puzzle(
        category = category,
        difficulty = difficulty
    )

    session['current_hints_used'] = 0

    # Record the puzzle in the session history
    session['puzzle_history'].append({
        'puzzle_id': puzzle['puzzle_id'],
        'category': category,
        'difficulty': difficulty,
        'question': puzzle['question'],
        'started_at': datetime.now(timezone.utc).isoformat()
    })

    return puzzle

**20. RECORD ATTEMPTS**

In [23]:
def record_attempt(
    session,
    puzzle,
    player_answer,
    is_correct,
    time_taken,
    attempts,
    hints_used
):
    """
    Record a player's attempt and update the session statistics.

    Stores individual attempt details in attempt_history and updates the overall session 
    performance metrics.
    """

    # Create a record containing the details of this attempt.
    attempt_record = {
        "attempt_id": str(uuid.uuid4()),
        "puzzle_id": puzzle['puzzle_id'],
        "category": puzzle['category'],
        "difficulty": puzzle['difficulty'],
        "player_answer": player_answer,
        "is_correct": is_correct,
        "time_taken": time_taken,
        "attempts": attempts,
        "hints_used": hints_used,
        "timestamp": datetime.now(timezone.utc).isoformat()
    }

    #Store individual attempt
    session['attempt_history'].append(attempt_record)

    #Update overall session statistics
    session['total_attempts'] += 1
    session['total_time'] += time_taken
    session['total_hints_used'] += hints_used

    if is_correct:
        session['correct_answers'] += 1
    else:
        session['incorrect_answers'] += 1

    return attempt_record

**21. RECORD PLAYER PERFORMANCE**

In [24]:
def get_player_performance(session):

    """
    Calculate overall player performance for the current session.
    Returns accuracy, average completion time, and average hints used. 
    """
    
    total = session['total_attempts']
    correct = session['correct_answers']

    # Calculate overall accuracy
    accuracy = (correct/total) * 100 if total > 0 else 0

    # Calculate average time per completed attempt
    average_time = session['total_time'] / total if total > 0 else 0
    
    # Calculate average hints used per completed attempt
    average_hints = session['total_hints_used'] / total if total > 0 else 0
    
    return {
        "total_attempts": total,
        "correct_answers": correct,
        "incorrect_answers": session['incorrect_answers'],
        "accuracy_percentage": round(accuracy, 2),
        "average_time_seconds": round(average_time, 2),
        "average_hints_used": round(average_hints, 2)
    }

**22. RETRIEVE RECENT PERFORMANCE**

In [25]:
#track recent performance
def get_recent_performance(session, limit=5):

    """
    Calculate player performance over the most recent attempts.

    Args:
        session: Current game session
        limit: Maximum number of recent attempts to consider

    Returns:
        Recent attempt records and their accuracy percentage
    """

    # Get the most recent attempts
    recent_attempts = session['attempt_history'][-limit:]

    # Handle sessions with no recorded attempts
    if not recent_attempts:
        return{
            "attempts": [],
            "accuracy_percentage": 0
        }

    # Count the correct answers among recent attempts
    correct_count = sum(
        1 for attempt in recent_attempts
        if attempt['is_correct']
    )

    # Calculate accuracy based on the recent attempts
    accuracy = (correct_count / len(recent_attempts)) *100

    return {
        "attempts": recent_attempts,
        "accuracy_percentage": round(accuracy,2)
    }

**23. RECORD COMPLETED PUZZLE**

In [26]:
def record_completed_puzzle(
    session,
    puzzle,
    final_answer,
    is_correct,
    time_taken,
    attempts,
    hints_used
):

    """
    Update the existing puzzle history record with completion details
    """

    # Find the puzzle that was started in the current session
    puzzle_record = None

    for record in session['puzzle_history']:
        if record['puzzle_id'] == puzzle['puzzle_id']:
            puzzle_record = record
            break

    # Ensure the puzzle exists in the session history
    if puzzle_record is None:
        raise ValueError('Puzzle not found in session history')
        
    # Add the completed puzzle to the session history
    puzzle_record.update({
        "final_answer": final_answer,
        "is_correct": is_correct,
        "time_taken": time_taken,
        "attempts": attempts,
        "hints_used": hints_used,
        "completed_at": datetime.now(timezone.utc).isoformat()
    })
    
    return puzzle_record

# ADAPTIVE DIFFICULTY

**24. BUILD ADAPTATION CONTEXT**

In [27]:
def build_adaptation_context(session, recent_limit=5):

    """
    Build a summary of the player's performance for AI-based difficulty adaptation.

    Parameters:
        session (dict): Current game session
        recent_limit (int): Number of recent completed puzzles to consider

    Returns:
        dict: Player performance context for the AI
    """

    # Get only puzzles that have been completed
    completed_puzzles = []

    for puzzle in session['puzzle_history']:
        if 'is_correct' in puzzle:
            completed_puzzles.append(puzzle)

    # Get the most recent completed puzzles
    recent_puzzles = completed_puzzles[-recent_limit:]

    # Calculate the overall performance
    total_completed = len(completed_puzzles)

    correct_count = sum(
        1 for puzzle in completed_puzzles
        if puzzle['is_correct']
    )

    accuracy = correct_count / total_completed * 100 if total_completed > 0 else 0

    # Calculate recent performance
    recent_correct = sum(
        1 for puzzle in recent_puzzles
        if puzzle['is_correct']
    )

    recent_accuracy = (
        recent_correct/len(recent_puzzles) * 100
        if recent_puzzles
        else 0
    )

    return{
        "current_difficulty": session['current_difficulty'],
        "total_puzzles_completed": total_completed,
        "overall_accuracy_percentage": round(accuracy, 2),
        "recent_accuracy_percentage": round(recent_accuracy, 2),
        "recent_puzzles": recent_puzzles
    }

**25. GET ADAPTATION PROMPT**

In [28]:
#adaptation prompt function
def get_adaptation_prompt(context):

    """
    Build the prompt for Gemini to recommend the next puzzle difficulty.

    Parameters:
        context (dict): Player performance data from build_adaptation_context()

    Returns:
        str: Prompt for Gemini
    """
    
    prompt = f"""
    You are the adaptive difficulty manager for Puzzle Pirates, an AI-powered brain puzzle game.
    Your job is to analyze player performance and recommend the difficulty of the next puzzle.

    Available difficulty levels:
    - easy
    - medium
    - hard

    Player performance data:
    {json.dumps(context, indent=2)}

    Decision guidelines:

    1. Consider both overall and recent performance.
    2. Consider correctness, completion time, attempts, and hints used from the recent puzzle records.
    3. Consistently strong performance may justify increasing difficulty.
    4. Repeated mistakes or substantial use of hints may justify decreasing difficulty.
    5. Mixed performance may justify maintaining current difficulty.
    6. Do not make a large jump without sufficient evidence.
    7. If the player has completed very few puzzles, be conservative.
    8. Only recommend one difficulty level above or below the current level.

    Return ONLY valid JSON in this exact format:

    {{
        "recommended_difficulty":"easy",
        "decision":"maintain",
        "reasoning": "Brief explanation of your decision"
    }}

    The decision must be one of:
    - increase
    - decrease
    - maintain

    The recommended difficulty must be:
    - easy
    - medium
    - hard.

    Do not include Markdown or additional text
    """

    return prompt

**26. GET DIFFICULTY RECOMMENDATION**

In [29]:
def get_ai_difficulty_recommendation(session):

    """
    Ask Gemini to recommend the difficulty for the next puzzle based on the player's session
    performance.

    Parameters:
        session (dict): Current game session

    Returns:
        dict: Validated difficulty recommendation from Gemini
    """

    # Build the player's performance context
    context = build_adaptation_context(session)

    # Create the prompt for Gemini
    prompt = get_adaptation_prompt(context)

    # Generate and parse Gemini's JSON response
    recommendation = generate_json_response(prompt)

    #Validate the AI response
    recommended_difficulty = recommendation.get("recommended_difficulty")

    if recommended_difficulty not in DIFFICULTY_LEVELS:
        raise ValueError("Gemini returned an invalid difficulty")

    decision = recommendation.get('decision')

    if decision not in ["increase", "decrease", "maintain"]:
        raise ValueWrror("Gemini returned an invalid decision")

    reasoning = recommendation.get('reasoning')
    
    if not isinstance(reasoning, str) or not reasoning.strip():
        raise ValueError("Gemini did not provide reasoning")

    return recommendation
    

**27. UPDATE DIFFICULTY**

In [32]:
def update_session_difficulty(session, recommendation):
    
    """
    Update the session's current difficulty and record the AI's adaptation decision.

    Parameters:
        session (dict): Current game session
        recommendation (dict): Validated recommendation from Gemini.

    Returns:
        dict: Record of the difficulty adaptation
    """
    
    previous_difficulty = session['current_difficulty']
    new_difficulty = recommendation['recommended_difficulty']
    decision = recommendation['decision']

    # Get the position of each difficulty level
    current_index = DIFFICULTY_LEVELS.index(previous_difficulty)
    new_index = DIFFICULTY_LEVELS.index(new_difficulty)

    # Check that the difficulty changes by no more than one level
    if abs(new_index - current_index) > 1:
        raise ValueError('Difficulty cannot change by more than one level')

    # CHeck that the decision matches the recommended difficulty
    if decision == 'increase' and new_index <= current_index:
        raise ValueError('An increase secision must raise the difficulty')

    elif decision == 'decrease' and new_index >= current_index:
        raise ValueError('A decrease decision must lower the difficulty')

    elif decision == 'maintain' and new_index != current_index:
        raise ValueError('A maintain decision must keep the same difficulty')

    # Update the session difficulty
    session['current_difficulty'] = new_difficulty

    # Record the adaptation decision
    adaptation_record = {
        "previous_difficulty": previous_difficulty,
        "new_difficulty": new_difficulty,
        "decision": recommendation['decision'],
        "reasoning": recommendation['reasoning'],
        "timestamp": datetime.now(timezone.utc).isoformat()
    }

    session['adaptation_history'].append(adaptation_record)

    return adaptation_record

# AGENT TOOLS

**28. CHECK ANSWERS**

In [40]:
#checking answers

def normalize_text(answer):

    """
    Standardize a text answer before comparison.

    Converts the answer to lowercase and removes leading, trailing, and extra whitespace.

    Parameters:
        answer: Player's submitted answer

    Returns:
        str: Normalized answer, or an empty string if input is not text
    """
    
    if not isinstance(answer, str):
        return ""

    return " ".join(answer.strip().lower().split())

def check_answer(puzzle, player_answer):

    """
    Check whether the player's answer matches the puzzle's correct answer.

    Supports integer, string, and structured answer types.

    Parameters:
        puzzle (dict): Generated puzzle containing the correct answer
        player_answer: Answer submitted by the player

    Returns:
        dict: Answer evaluation containing correctness, feedback, submitted answer, and explanation
        when correct
    """
    
    answer_type = puzzle['answer_type']
    correct_answer = puzzle['correct_answer']

    is_correct = False

    # --------------------------------------------------
    # 1. Check integer answers (number pattern puzzles)
    # --------------------------------------------------
    if answer_type == 'integer':
        try:
            #Reject boolean valyes because bool is a subclass of int
            if isinstance(player_answer, bool):
                raise ValueError

            # Convert the submitted answer to an integer
            submitted_answer = int(player_answer)
            
            # Reject decimals and non-integer representations
            if str(player_answer).strip() not in [str(submitted_answer), f"+{submitted_answer}"]:
                raise ValueError

            # Compare the submitted integer with the correct answer
            is_correct = submitted_answer == correct_answer

        except(ValueError, TypeError):
            is_correct = False

    # --------------------------------------------------
    # 2. Check string answers (riddle puzzles)
    # --------------------------------------------------
    elif answer_type == 'string':

        # Normalize the player's answer
        normalized_answer = normalize_text(player_answer)

        # Include the corrext answer and accepted alternatives
        accepted_answers = [correct_answer] + puzzle.get('accepted_answers', [])

        # Normalize all valid answers before comparison
        normalized_accepted = [normalize_text(answer) for answer in accepted_answers
            if isinstance(answer, str)]

        is_correct = normalized_answer in normalized_accepted

    # --------------------------------------------------
    # 3. Check structured answers (reserved for future use)
    # --------------------------------------------------
    elif answer_type == 'structured':
        is_correct = player_answer == correct_answer

    # --------------------------------------------------
    # 4. Handle unsupported answer types
    # --------------------------------------------------
    else: 
        raise ValueError(f"Unsupported answer type: {answer_type}")

    # --------------------------------------------------
    # 5. Prepare the evaluation result
    # --------------------------------------------------
    result = {
        "is_correct": is_correct,
        "player_answer": player_answer,
        "feedback": (
            "Correct! Well done!"
            if is_correct
            else "Not quite. Try again!"
        ),
        "explanation": (
            puzzle["explanation"]
            if is_correct
            else None
        )
    }

    return result

**29. HINTS**

In [52]:
def provide_hint(puzzle, session):

    """
    Provide the next available hint for a puzzle.

    Parameters:
        puzzle (dict): Current puzzle containing the list of hints.
        session (dict): Current game session

    Returns:
        dict: Hint result containing the hint, updated hint count, and availability status
    """

    # Get the number of hints already used in this round
    hints_used = session['current_hints_used']
    
    # Get the available hints from the puzzle
    hints = puzzle['hints']

    # Check whether the player has reached the hint limit
    if hints_used >= MAX_HINTS:
        return{
            "hint": None,
            "hints_used": hints_used,
            "remaining_hints": 0,
            "message": "You have used all available hints"
        }

    # Get the next hint based on the number already used
    hint = hints[hints_used]

    # Update the number of hints used
    session['current_hints_used'] += 1
    session['total_hints_used'] += 1

    # Return the hint and updated hint information
    return {
        "hint": hint,
        "hints_used": session['current_hints_used'],
        "remaining_hints": MAX_HINTS - session['current_hints_used'],
        "message": "Here is your next hint!"
    }

**30. EXPLAIN PUZZLE**

In [54]:
def explain_puzzle(puzzle):
    """
    Return the explanation for a puzzle.

    Parameters:
        puzzle (dict): Current puzzle containing its explanation.

    Returns:
        dict: Explanation and a message.
    """

    explanation = puzzle.get("explanation")

    # Check whether an explanation is available
    if not isinstance(explanation, str) or not explanation.strip():
        return {
            "explanation": None,
            "message": "No explanation is available for this puzzle."
        }

    return {
        "explanation": explanation,
        "message": "Here is the puzzle explanation."
    }

# GEMINI TOOL SCHEMAS

In [55]:
generate_puzzle_tool = {
    "name": "generate_puzzle",
    "description": ("Generate a new validated puzzle for the player"
                   "using the requested category and difficulty."),
    "parameters": {
        "type": 'OBJECT',
        "properties": {
            "category": {
                "type": "STRING",
                "enum": PUZZLE_CATEGORIES,
                "description": "The category of the puzzle to generate."
            },
            "difficulty": {
                "type": "STRING",
                "enum": DIFFICULTY_LEVELS,
                "description": "The difficulty level of the puzzle"
            }
        },
        "required": ["category", "difficulty"]
    }
}

In [56]:
check_answer_tool = {
    "name": "check_answer",
    "description": ("Check the player's submitted answer against"
                   "the current puzzle's correct answer."),
    "parameters": {
        "type": "OBJECT",
        "properties": {
            "player_answer": {
                "type": "STRING",
                "description": "The answer submitted by the player"
            }
        },
        "required": ["player_answer"]
    }
}

In [57]:
provide_hint_tool = {
    "name": "provide_hint",
    "description": ("Provide the player with the next available hint"
                   "for the current puzzle"),
    "parameters": {
        "type": "OBJECT",
        "properties": {},
        "required": []
    }
}

In [58]:
explain_puzzle_tool = {
    "name": "explain_puzzle",
    "description": ("Provide the player with the explanation"
                   "of the current puzzle"),
    "parameters": {
        "type": "OBJECT",
        "properties": {},
        "required": []
    }
}

In [59]:
get_performance_tool = {
    "name": "get_player_performance",
    "description": ("Retrieve the player's overall performance statistics for the current game"
                   "session, including accuracy, completion time, and hints used"),
    "parameters": {
        "type": "OBJECT",
        "properties": {},
        "required": []
    }
}

In [60]:
adapt_difficulty_tool = {
    "name": "adapt_difficulty",
    "description": ("Analyze the player's performance and adapt the difficulty of the future puzzles"
                   "Increase, decrease, or maintain the current difficulty based on the"
                   "player's performance"),
    "parameters": {
        "type": "OBJECT",
        "properties": {},
        "required": []
    }
}

In [61]:
agent_tools = [
    generate_puzzle_tool,
    check_answer_tool,
    provide_hint_tool,
    explain_puzzle_tool,
    get_performance_tool,
    adapt_difficulty_tool
]

**TOOL EXECUTION LAYER**

In [62]:
def execute_agent_tool(tool_name, tool_args, session, current_puzzle=None):
    """
    Execute the Python function requested by the AI agent.

    Parameters:
        tool_name (str): Name of the requested tool
        tool_args (dict): Arguments supplied by Gemini
        session (dict): Current game session
        current_puzzle (dict): Current puzzle, if available

    Returns:
        dict: Result produced by the requested tool
    """
    # 1. Generate a new puzzle
    if tool_name == 'generate_puzzle':
        category = tool_args.get('category')
        difficulty = tool_args.get('difficulty')

        puzzle = generate_validated_puzzle(category, difficulty)

        return{
            "success": True,
            "tool": tool_name,
            "result": puzzle
        }

    # 2. Check the player's answer
    elif tool_name == 'check_answer':
        if current_puzzle is None:
            return {
                "success": False,
                "error": "There is no active puzzle to check"
            }

        player_answer = tool_args.get('player_answer')

        result = check_answer(current_puzzle, player_answer)

        return {
            "success": True,
            "tool": tool_name,
            "result": result
        }

    # 3. Provide the next available hint
    elif tool_name == "provide_hint":
        if current_puzzle is None:
            return{
                "success": False,
                "error": "There is no active puzzle"
            }

        result = provide_hint(current_puzzle, session)

        return {
            "success": True,
            "tool": tool_name,
            "result": result
        }

    # 4. Explain the current puzzle
    elif tool_name == "explain_puzzle":
        if current_puzzle is None:
            return {
                "success": False,
                "error": "There is no active puzzle"
            }

        result = explain_puzzle(current_puzzle)

        return {
            "success": True,
            "tool": tool_name,
            "result": result
        }

    # 5. Retrieve the player performance statistics
    elif tool_name == "get_player_performance":
        result = get_player_performance(session)

        return {
            "success": True,
            "tool": tool_name,
            "result": result
        }

    # 6. Adapt puzzle difficulty based on player performance
    elif tool_name == "adapt_difficulty":
        recommendation = get_ai_difficulty_recommendation(session)

        adaptation = update_session_difficulty(session, recommendation)

        return {
            "success": True,
            "tool": tool_name,
            "result": {
                "recommendation": recommendation,
                "adaptation": adaptation
            }
        }

    # Handle unsupported tool requests
    else:
        return {
            "success": False,
            "error": f"Unknown tool: {tool_name}"
        }

In [42]:
number_puzzle = {
    "puzzle_id": str(uuid.uuid4()),
    "category": "number_pattern",
    "difficulty": "easy",
    "question": "Find the missing number.",
    "content": {
        "sequence": [2, 4, 6, None, 10],
        "missing_position": 3,
        "pattern_type": "arithmetic"
    },
    "answer_type": "integer",
    "correct_answer": 8,
    "accepted_answers": [],
    "explanation": "The sequence increases by 2 each time.",
    "hints": [
        "Look at the difference between consecutive numbers.",
        "The numbers increase by the same amount.",
        "Try adding 2 to the previous number."
    ]
}

riddle_puzzle = {
    "puzzle_id": str(uuid.uuid4()),
    "category": "riddle",
    "difficulty": "easy",
    "question": "What am I?",
    "content": {
        "riddle_text": "I follow you during the day but disappear in darkness. What am I?"
    },
    "answer_type": "string",
    "correct_answer": "shadow",
    "accepted_answers": [],
    "explanation": "A shadow is formed when an object blocks light.",
    "hints": [
        "Look at the ground on a sunny day.",
        "It is created when light is blocked.",
        "It follows your movements."
    ]
}

In [41]:
#logic puzzle validation

def validate_logic(puzzle):
    errors = []

    # Check required puzzle fields
    required_fields = [
        "content",
        "answer_type",
        "correct_answer",
        "hints"
    ]

    for field in required_fields:
        if field not in puzzle:
            errors.append(f"Logic puzzle is missing: {field}")

    if errors:
        return errors

    content = puzzle["content"]

    # Check required content fields
    required_content_fields = [
        "scenario",
        "left_group",
        "right_group",
        "clues"
    ]

    for field in required_content_fields:
        if field not in content:
            errors.append(f"Logic puzzle content is missing: {field}")

    if errors:
        return errors

    # Validate scenario
    if not isinstance(content["scenario"], str) or not content["scenario"].strip():
        errors.append("Logic scenario must be non-empty text")

    # Validate clues
    clues = content["clues"]

    if not isinstance(clues, list) or not clues:
        errors.append("Logic puzzle must contain at least one clue")

    elif not all(isinstance(clue, str) and clue.strip() for clue in clues):
        errors.append("Every logic clue must be non-empty text")

    # Validate groups
    left_group = content["left_group"]
    right_group = content["right_group"]

    if not isinstance(left_group, list) or not left_group:
        errors.append("Left group must be a non-empty list")

    if not isinstance(right_group, list) or not right_group:
        errors.append("Right group must be a non-empty list")

    if isinstance(left_group, list) and isinstance(right_group, list):

        # Check for empty or invalid entities
        if not all(isinstance(item, str) and item.strip() for item in left_group):
            errors.append("Every item in left group must be non-empty text")

        if not all(isinstance(item, str) and item.strip() for item in right_group):
            errors.append("Every item in right group must be non-empty text")

        # Check duplicates
        if len(left_group) != len(set(left_group)):
            errors.append("Left group contains duplicate entities")

        if len(right_group) != len(set(right_group)):
            errors.append("Right group contains duplicate entities")

        # Groups must be equal in size
        if len(left_group) != len(right_group):
            errors.append("Left and right groups must have the same number of entities")

    # Validate answer type
    if puzzle["answer_type"] != "structured":
        errors.append("Logic puzzle answer type must be structured")

    # Validate correct answer
    answer = puzzle["correct_answer"]

    if not isinstance(answer, dict):
        errors.append("Logic correct answer must be a dictionary")

    elif isinstance(left_group, list) and isinstance(right_group, list):

        # Every left entity must have an answer
        if set(answer.keys()) != set(left_group):
            errors.append("Correct answer must include every left-group entity exactly once")

        # Every answer must belong to the right group
        if not all(value in right_group for value in answer.values()):
            errors.append("Correct answer contains an entity not in the right group")

        # Each right entity can only be assigned once
        if len(answer.values()) != len(set(answer.values())):
            errors.append("Correct answer assigns the same entity more than once")

    # Validate hints
    hints = puzzle["hints"]

    if not isinstance(hints, list) or len(hints) != 3:
        errors.append("Logic puzzle must contain exactly 3 hints")

    elif not all(isinstance(hint, str) and hint.strip() for hint in hints):
        errors.append("Every hint must be non-empty text")

    return errors

In [15]:
#final validation function
def validate_puzzle(puzzle):
    errors = validate_common_fields(puzzle)

    if errors:
        return{
            'is_valid': False,
            'errors': errors
        }

    category = puzzle['category']

    if category == 'number_pattern':
        errors.extend(validate_number_pattern(puzzle))

    elif category == 'riddle':
        errors.extend(validate_riddle(puzzle))

    elif category == 'logic':
        errors.extend(validate_logic(puzzle))

    return{
        'is_valid': len(errors) ==0,
        'errors': errors
    }
    

In [35]:
def generate_validated_number_pattern(
    difficulty,
    max_retries=3
):

    feedback = None

    for attempt in range(1, max_retries + 1):

        print(f"Generation attempt {attempt}/{max_retries}")

        try:
            # Generate puzzle using Gemini
            generated_data = generate_number_pattern(
                difficulty,
                feedback=feedback
            )

            # Convert into our common puzzle structure
            puzzle = convert_to_puzzle(generated_data)

            # Validate the puzzle
            validation_result = validate_puzzle(puzzle)

            if validation_result["is_valid"]:

                puzzle["validation_status"] = "passed"

                print("Puzzle generated and validated successfully!")

                return puzzle

            # Prepare feedback for the next generation attempt
            feedback = "\n".join(
                validation_result["errors"]
            )

            print("Validation failed:")

            for error in validation_result["errors"]:
                print("-", error)

        except (ValueError, KeyError, TypeError, json.JSONDecodeError) as error:

            feedback = str(error)

            print("Generation or parsing error:", error)

    raise RuntimeError(
        f"Could not generate a valid puzzle after {max_retries} attempts."
    )

In [36]:
generated_puzzle_history = []


In [78]:
# for i in range(3):

#     generated_data = generate_number_pattern("hard")

#     print(f"\nPuzzle {i + 1}")
#     print("Sequence:", generated_data["content"]["sequence"])
#     print("Pattern:", generated_data["content"]["pattern_type"])
#     print("Answer:", generated_data["correct_answer"])

#     puzzle = convert_to_puzzle(generated_data)

#     result = validate_puzzle(puzzle)

#     if result["is_valid"]:
#         generated_puzzle_history.append({
#             "sequence": puzzle["content"]["sequence"],
#             "pattern_type": puzzle["content"]["pattern_type"],
#             "correct_answer": puzzle["correct_answer"]
#         })

In [52]:
def get_logic_puzzle_prompt(difficulty, recent_puzzles=None, feedback=None):
    recent_puzzles = recent_puzzles or []

    difficulty_guidelines = {
        "easy": """
        - Use exactly 3 entities in both left and right groups.
        - Use 3 to 4 simple clues.
        - Include mostly direct elimination clues.
        - The puzzle should be solvable with basic deduction.
        """,

        "medium": """
        - Use exactly 4 entities both in left and right groups.
        - Use 4 to 5 clues.
        - Combine direct clues with elimination reasoning.
        - Require several deduction steps.
        """,

        "hard": """
        - Use exactly 4 entities in both left and right groups.
        - Use 5 to 6 clues.
        - Require multiple deduction steps.
        - Clues may interact with each other, but every clue must be a simple equality or 
        inequality constraint. 
        """
    }

    if difficulty not in difficulty_guidelines:
        raise ValueError('Invalid difficulty level.')

    prompt = f"""
    Your are an expert logic puzzle creator for Puzzle Pirated, an adaptive brain puzzle game.

    Generate ONE original matching and deduction logic puzzle.

    DIFFICULTY: {difficulty}
    DIFFICULTY EXPECTATIONS: {difficulty_guidelines[difficulty]}

    PUZZLE REQUIREMENTS:

    1. Create two groups of entities.
    2. Each group must contain the same number of unique entities.
    3. Each entity in the first group must match exactly one entity in the second group. 
    4. Every entity in the second group must be matched exactly once.
    5. Provide a scenario that explains the puzzle.
    6. Provide clear clues that allow the player to determine every match. 
    7. The clues must be logically consistent and should not provide a direct answer.
    8. There must be exactly ONE valid solution.
    9. The correct answer must match the question and the clues.
    10. The correct answer must be a complete mapping.
    11. Generate exactly 3 progressive hints.
    12. Hints should guide the player's reasoning without directly revealing the complete solution.
    13. Include a clear explanation of the solution.
    14. Avoid trivia or outside knowledge.
    15. Make the puzzle original and avoid repeating recent puzzles.

    VARIETY REQUIREMENTS:

    - Explore different themes and scenarios.
    - Vary the names and entities.
    - Vary the types of clues and deduction steps.
    - Avoid copying recently generated puzzles.

    RECENTLY GENERATED PUZZLES TO AVOID:
    {json.dumps(recent_puzzles, indent=2)}

    Return ONLY valid JSON using these fields:
    {{
        "category": "logic",
        "difficulty": "{difficulty}",
        "question": "The puzzle question shown to the player.",
        "content": {{
            "scenario": "Scenario description.",
            "left_group": ["Person A", "Person B", "Person C"],
            "right_group": ["Item A", "Item B", "Item C"],
            "clues": [
                "First clue",
                "Second clue",
                "Third clue"
            ]
        }},
        "answer_type": "structured",
        "correct_answer": {{
            "Person A": "Item B",
            "Person B": "Item C",
            "Person C": "Item A"
        }},
        "options": [],
        "accepted_answers": [],
        "hints": [
            "First hint",
            "Second hint",
            "Third hint"
        ],
        "explanation": "Explain the deductions leading to the solution."
    }}

    The example values illustrate the required structure only.
    Generate a new puzzle and a logically consistent solution. 

    Return only valid JSON. Do not include Markdown. 
    
    """

    if feedback:
        prompt += f"""

        PREVIOUS VALIDATION ERRORS: {feedback}

        Generate a new puzzle that corrects these issues.
        """

    return prompt

In [40]:
def generate_logic_puzzle(difficulty, recent_puzzles=None, feedback=None):
    prompt = get_logic_puzzle_prompt(difficulty, 
                                    recent_puzzles=recent_puzzles,
                                    feedback=feedback)

    response = client.models.generate_content(
        model = MODEL_NAME,
        contents = prompt,
        config = {
            "response_mime_type": "application/json",
            "temperature": 0.9
        }
    )

    if not response.text:
        raise ValueError('Gemini returned an empty response')

    generated_data = json.loads(response.text)

    return generated_data

In [57]:
def validate_puzzle(puzzle):
    category = puzzle.get('category')

    if category == "number_pattern":
        errors = validate_number_pattern(puzzle)

    elif category == "riddle":
        errors = validate_riddle(puzzle)

    else:
        errors = [f"Unsupported category: {category}"]

    return errors